# 🏆 [Day 35] 실전 벡터 검색 & GraphRAG 핸즈온 워크북

> **핵심 학습 목표**:
> 텍스트의 의미적 유사도를 탐색하는 **벡터 검색(Vector Search)**과 개체 간의 네트워크 구조를 분석하는 **지식그래프(Knowledge Graph)**를 결합하여, 단순 Chunk 검색의 한계를 돌파하는 **GraphRAG** 아키텍처를 직접 구축하고 실측합니다.
>
> 1. 🧐 **[생각하기 1]**: 왜 Naive RAG(단순 청크 벡터 검색)는 다단계 관계형 질문에서 환각을 일으키는가?
> 2. ⚡ **[인덱스 비교]**: `SEARCH n IN (VECTOR INDEX doc_vec ...)` vs Lucene 전문 검색(`doc_ft`)의 장단점 실측
> 3. 🔗 **[다리 놓기]**: 왜 논문과 개체를 이름이 아니라 `id`로 매칭하여 `:MENTIONS`로 연결해야 하는가?
> 4. 🧠 **[GDS 연계]**: `drugGraph` 투영 위에서 PageRank 중심성과 Leiden 커뮤니티 산출
> 5. 🎯 **[하이브리드 리랭킹]**: $Score = (1-w) \cdot Sim_{norm} + w \cdot PageRank_{norm}$ 가중 융합 실측
> 6. 🔬 **[커뮤니티 스코핑]**: 기준 약물(`Clopidogrel`)의 Leiden 군집으로 검색 문맥 좁히기
> 7. 🤖 **[LLM QA & 인용 검증]**: `gpt-4o-mini`로 답변 생성 시 `[PMCxxxxxxx]` 출처 대조 및 '보고했다' vs '입증됐다' 사실 엄밀성 검증

## 0. 환경 설정 및 Neo4j GDS / OpenAI 연결 (로컬 7689 포트)

In [1]:
import os
import sys
import json
import gzip
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1) 접속 정보 읽기: Day 35 실습 전용 로컬 인스턴스 (7689)
env_file = os.path.join(os.path.abspath(''), '.env')
if os.path.exists(env_file):
    load_dotenv(env_file, override=True)

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'test0011')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=768)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print('✅ [연결 성공] Neo4j URI:', NEO4J_URI)
print('✅ [임베딩 모델]: text-embedding-3-large (768차원)')
print('✅ [LLM 모델]: gpt-4o-mini')

✅ [연결 성공] Neo4j URI: bolt://localhost:7689
✅ [임베딩 모델]: text-embedding-3-large (768차원)
✅ [LLM 모델]: gpt-4o-mini


## 1. 현재 적재된 지식그래프 및 논문 데이터 무결성 점검
- `:Compound`, `:Disease`, `:Gene`, `:Symptom`, `:PharmacologicClass` 노드
- `:Document` 논문 노드 69편
- `:MENTIONS` 논문-개체 간 연결 다리

In [2]:
node_counts = run_cypher('''
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS count
ORDER BY count DESC
''')
print('=== [Neo4j 노드 현황] ===')
for r in node_counts:
    print(f"  - :{r['label']:20} {r['count']:>6,}개")

rel_counts = run_cypher('''
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(r) AS count
ORDER BY count DESC
''')
print('\n=== [Neo4j 관계 현황] ===')
for r in rel_counts[:6]:
    print(f"  - [:{r['rel_type']:18}] {r['count']:>6,}건")
print(f"  ... 외 총 {len(rel_counts)}개 관계 유형")

=== [Neo4j 노드 현황] ===
  - :Gene                 13,113개
  - :Compound              1,531개
  - :Symptom                 415개
  - :PharmacologicClass      345개
  - :Disease                 136개
  - :Document                 69개

=== [Neo4j 관계 현황] ===
  - [:DOWNREGULATES_CG  ] 21,102건
  - [:UPREGULATES_CG    ] 18,756건
  - [:ASSOCIATES        ] 12,623건
  - [:BINDS             ] 11,571건
  - [:UPREGULATES_DG    ]  7,731건
  - [:DOWNREGULATES_DG  ]  7,623건
  ... 외 총 13개 관계 유형


## 2. 벡터 인덱스 vs 전문 인덱스 비교 탐색
> **Thinking Point 1**: 유전자 기호 `CYP3A4`나 약물 코드 같은 고유명사는 벡터 검색이 유리할까요, 전문 인덱스(Fulltext)가 유리할까요?

In [3]:
keyword = 'CYP3A4'

# 1) 전문 인덱스 (Lucene Fulltext)
ft_hits = run_cypher('''
CALL db.index.fulltext.queryNodes('doc_ft', $term) YIELD node, score
RETURN node.pmcid AS pmcid, node.title AS title, score
ORDER BY score DESC LIMIT 3
''', term=f'"{keyword}"')

# 2) 벡터 인덱스 (HNSW Cosine)
q_vec = embedder.embed_query(keyword)
vec_hits = run_cypher('''
MATCH (n:Document)
SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 3) SCORE AS score
RETURN n.pmcid AS pmcid, n.title AS title, score
ORDER BY score DESC
''', q=q_vec)

print(f"🔍 [전문 인덱스 키워드 '{keyword}' 검색 결과]")
for idx, h in enumerate(ft_hits, 1):
    print(f"  {idx}. [{h['pmcid']}] score: {h['score']:.4f} | {h['title'][:55]}...")

print(f"\n🔍 [벡터 인덱스 키워드 '{keyword}' 검색 결과]")
for idx, h in enumerate(vec_hits, 1):
    print(f"  {idx}. [{h['pmcid']}] score: {h['score']:.4f} | {h['title'][:55]}...")

🔍 [전문 인덱스 키워드 'CYP3A4' 검색 결과]
  1. [PMC13474907] score: 1.9998 | Factor XI Inhibition for Ischemic Stroke Secondary Prev...
  2. [PMC13368705] score: 1.7835 | Pharmacogenetic analyses in people with dementia in Nor...
  3. [PMC13494692] score: 1.6094 | Pyridyl‐Thiazole‐Thiosemicarbazones as Antitrypanosomat...

🔍 [벡터 인덱스 키워드 'CYP3A4' 검색 결과]
  1. [PMC13494176] score: 0.7049 | Proteomic profiling of metabolizing enzymes and transpo...
  2. [PMC13464512] score: 0.6947 | Tamoxifen Therapy Is Associated With Altered Intestinal...
  3. [PMC13432136] score: 0.6911 | Pharmacogenomic landscape in Thailand: Array-based prof...


## 3. GDS PageRank 기반 하이브리드 리랭킹 (MinMax Fusion)
> **수식**: $Score_{fused} = (1 - w) \cdot Sim_{norm} + w \cdot PageRank_{norm}$
> 
> - 벡터 유사도 점수와 그래프 중심성 점수의 눈금(Scale) 차이를 맞추기 위해 0~1 정규화를 수행합니다.

In [5]:
# 1) 논문이 언급한 개체들의 PageRank 평균을 d.graph_score에 저장
run_cypher('''
MATCH (d:Document)-[:MENTIONS]->(e)
WHERE e.pagerank IS NOT NULL
WITH d, avg(e.pagerank) AS mean
SET d.graph_score = mean
''')

# 2) PageRank가 없는 개체만 언급한 논문에는 기본 바닥값(0.15) 부여
run_cypher('''
MATCH (d:Document) 
WHERE d.graph_score IS NULL 
SET d.graph_score = 0.15
''')



[]

In [6]:
question = '약물 대사 유전자 검사를 처방 전에 하면 무엇이 달라지나요?'
q_vec = embedder.embed_query(question)

candidates = run_cypher('''
MATCH (n:Document)
SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 5) SCORE AS score
RETURN n.pmcid AS pmcid, n.title AS title, score, n.graph_score AS graph_score
ORDER BY score DESC
''', q=q_vec)

def minmax(vals):
    lo, hi = min(vals), max(vals)
    return [1.0] * len(vals) if hi == lo else [(v - lo) / (hi - lo) for v in vals]

sim_norm = minmax([c['score'] for c in candidates])
graph_norm = minmax([c['graph_score'] for c in candidates])

w = 0.3
fused = [(1 - w) * s + w * g for s, g in zip(sim_norm, graph_norm)]

print(f"🎯 [질문]: '{question}'\n")
print('순위 | PMCID | 원본 유사도 | 그래프 점수 | 융합 스코어 (w=0.3)')
print('-'*70)
sorted_items = sorted(zip(candidates, fused), key=lambda x: -x[1])
for rank, (cand, f_score) in enumerate(sorted_items, 1):
    print(f"{rank:>4} | {cand['pmcid']:10} | {cand['score']:11.4f} | {cand['graph_score']:11.4f} | {f_score:11.4f}")

🎯 [질문]: '약물 대사 유전자 검사를 처방 전에 하면 무엇이 달라지나요?'

순위 | PMCID | 원본 유사도 | 그래프 점수 | 융합 스코어 (w=0.3)
----------------------------------------------------------------------
   1 | PMC13494111 |      0.7375 |      0.1500 |      1.0000
   2 | PMC13368705 |      0.7296 |      0.1500 |      0.8263
   3 | PMC13432136 |      0.7108 |      0.1500 |      0.4151
   4 | PMC13473732 |      0.7061 |      0.1500 |      0.3121
   5 | PMC13464512 |      0.7056 |      0.1500 |      0.3000


## 4. Leiden 커뮤니티 기반 문맥 스코핑 (Context Scoping)
> 기준 약물(`Clopidogrel`)이 속한 네트워크 클러스터의 개체들을 언급한 논문으로 검색 풀을 한정하여 주제 이탈을 방지합니다.

In [7]:
anchor_name = 'Clopidogrel'
comm_res = run_cypher('MATCH (a {name: $name}) RETURN a.community AS c', name=anchor_name)
target_comm = comm_res[0]['c']

scoped_hits = run_cypher('''
MATCH (n:Document)
SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 20) SCORE AS score
MATCH (n)-[:MENTIONS]->(e)
WHERE e.community = $c
RETURN DISTINCT n.pmcid AS pmcid, n.title AS title, score
ORDER BY score DESC
''', q=q_vec, c=target_comm)

print(f"🏢 기준 약물 '{anchor_name}' 커뮤니티 ID: {target_comm}")
print(f"🎯 동일 커뮤니티 개체를 언급한 관련 논문 필터링 결과: 총 {len(scoped_hits)}편\n")
for idx, h in enumerate(scoped_hits[:5], 1):
    print(f"  {idx}. [{h['pmcid']}] score: {h['score']:.4f} | {h['title'][:60]}...")

🏢 기준 약물 'Clopidogrel' 커뮤니티 ID: None
🎯 동일 커뮤니티 개체를 언급한 관련 논문 필터링 결과: 총 0편



## 5. LangChain + GPT-4o-mini GraphRAG 근거 기반 질의응답
- 발췌문 + 언급 개체 조합 프롬프트
- `[PMCxxxxxxx]` 인용 표기 및 사실 엄밀성 검증

In [8]:
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 의학 바이오 지식그래프 기반 전문 연구 보조 AI입니다.\n'
               '반드시 주어진 [논문 발췌] 내용에만 엄격히 근거하여 답변하세요.\n'
               '발췌문에 명시되지 않은 사실은 절대 지어내지 마세요 (환각 엄금).\n'
               '답변의 각 문장 끝에는 반드시 근거가 된 논문의 [PMCxxxxxxx] 식별자를 명시하세요.'),
    ('human', '다음 논문 발췌를 근거로 질문에 한국어 세 문장 이내로 답변하세요.\n\n'
              '[논문 발췌문]\n{context}\n\n'
              '[질문]: {question}')
])
qa_chain = qa_prompt | llm | StrOutputParser()

top_pmcids = [item[0]['pmcid'] for item in sorted_items[:3]]

rows = run_cypher('''
MATCH (d:Document) WHERE d.pmcid IN $ids
OPTIONAL MATCH (d)-[:MENTIONS]->(e)
WITH d, e ORDER BY e.name
RETURN d.pmcid AS pmcid, d.title AS title, d.text AS text,
       collect(DISTINCT e.name)[..6] AS entities
''', ids=top_pmcids)
by_id = {r['pmcid']: r for r in rows}

context_parts = []
for pid in top_pmcids:
    if pid in by_id:
        r = by_id[pid]
        context_parts.append(f"[{r['pmcid']}] {r['title']}\n  본문: {r['text'][:350]}...\n  언급 개체: {', '.join(r['entities'])}")

final_context = '\n\n'.join(context_parts)
answer = qa_chain.invoke({'context': final_context, 'question': question})

print('📝 [최종 GraphRAG 모델 답변]\n')
print(answer)

📝 [최종 GraphRAG 모델 답변]

약물 대사 유전자 검사를 처방 전에 실시하면, 약물의 부작용을 30% 줄일 수 있는 가능성이 있습니다. 이는 유전자 기반의 용량 추천이 이루어질 수 있도록 하여, 환자 개개인의 유전적 특성에 맞춘 맞춤형 치료를 가능하게 합니다. 그러나 현재 유전자 검사가 일상적으로 사용되지 않고 있는 상황입니다 [PMC13368705].


---
## 🎓 자가 진단 퀴즈 (Self-Check Quiz)

1. **Q. Neo4j `SEARCH` 절에서 반환되는 점수의 범위와 의미는 무엇인가요?**
   - **A**: $0.0 \sim 1.0$ 사이의 값이며, 전통 코사인 유사도($-1 \sim 1$)를 $\frac{1 + \cos}{2}$로 변환하여 양수로 정규화한 값입니다.

2. **Q. 단순 벡터 검색 순위와 비교했을 때, PageRank 리랭킹을 적용하면 어떤 논문이 유리해지나요?**
   - **A**: 네트워크 내에서 많은 연결과 권위도를 갖는 핵심 화합물/질환 노드를 다수 언급한 논문의 순위가 상위로 보정(Promotion)됩니다.

3. **Q. DART 공시 시스템에서 본 GraphRAG 기법을 적용할 수 있는 영역은 어디인가요?**
   - **A**: 15,000건 공시 원문 텍스트에 대한 의미 기반 검색과, 지분 네트워크(5% 대량보유/최대주주)의 PageRank 및 계열사 커뮤니티를 결합한 '경영권 분쟁 및 지배구조 하이브리드 리트리버' 구축에 그대로 적용됩니다.